# 05. 전체 그래프 조립 + 실행

01~04번 노트북에서 각자 저장한 `src/nodes_b.py`, `src/nodes_cd.py`, `src/nodes_e.py`, `src/nodes_fgh.py`를 가져와
`A -> B -> {C,D,E} -> F -> G <-> H` 그래프로 조립하고 실제로 돌린다.

**먼저 01~04번 노트북을 각자 한 번씩 끝까지 실행해서 파일을 생성해둬야 한다.** 넷 중 하나라도 안 돌렸으면 이 노트북의 import가 실패한다.

In [ ]:
import sys
sys.path.insert(0, "..")

import os
from dotenv import load_dotenv
load_dotenv("../.env")

for key in ["OPENAI_API_KEY", "TAVILY_API_KEY"]:
    if not os.environ.get(key):
        print(f"경고: {key}가 .env에 없다. 아래 실제 실행 셀은 실패한다.")

## 1. 각 담당 노트북이 만든 노드 함수 가져오기

In [ ]:
from src.nodes_b import make_node_b
from src.nodes_cd import make_node_c, make_node_d
from src.nodes_e import make_node_e
from src.nodes_fgh import make_node_f, make_node_g, node_h_validate, route_after_h

print("네 파일 전부 import 성공")

## 2. A. 기술 선정 (Human 기반, 정적)

LLM 호출이 없어서 별도 노트북 없이 여기서 바로 정의한다.

In [ ]:
from src import config

def node_a_select_technologies(state):
    return {
        "selected_technologies": config.SELECTED_TECHNOLOGIES,
        "target_domain": config.TARGET_DOMAIN,
    }

## 3. 리트리버 · LLM · 웹서치 도구 준비

01번·03번 노트북에서 이미 만들어본 것과 같다 - 여기서 다시 한번 만든다(프로세스가 다르므로).

In [ ]:
from src.ingest import build_tech_retriever, build_domain_retriever

print("리트리버 구축 중 (Qwen3-Embedding-0.6B 다운로드가 처음엔 걸릴 수 있음)...")
tech_retriever = build_tech_retriever()
domain_retriever = build_domain_retriever()
print("완료")

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch

llm = init_chat_model(config.LLM_MODEL, model_provider=config.LLM_PROVIDER, temperature=config.LLM_TEMPERATURE)
web_search_tool = TavilySearch(max_results=5)

## 4. 그래프 조립

3-4절 mermaid 그대로. Reducer가 붙은 State 키는 하나도 없다 — 모든 키를 쓰는 노드가 정확히 1개씩이라 동시 쓰기 충돌 지점이 없기 때문(3-1절).

In [ ]:
from langgraph.graph import StateGraph, START, END
from src.state import GraphState

def build_graph(llm, tech_retriever, domain_retriever, web_search_tool):
    g = StateGraph(GraphState)

    g.add_node("A", node_a_select_technologies)
    g.add_node("B", make_node_b(llm, tech_retriever, web_search_tool))
    g.add_node("C", make_node_c(llm, web_search_tool))
    g.add_node("D", make_node_d(llm, web_search_tool))
    g.add_node("E", make_node_e(llm, domain_retriever))
    g.add_node("F", make_node_f(llm))
    g.add_node("G", make_node_g(llm))
    g.add_node("H", node_h_validate)

    g.add_edge(START, "A")
    g.add_edge("A", "B")
    g.add_edge("B", "C")
    g.add_edge("B", "D")
    g.add_edge("B", "E")
    g.add_edge("C", "F")
    g.add_edge("D", "F")
    g.add_edge("E", "F")
    g.add_edge("F", "G")
    g.add_edge("G", "H")
    g.add_conditional_edges("H", route_after_h, {"END": END, "G": "G"})

    return g.compile()

graph = build_graph(llm, tech_retriever, domain_retriever, web_search_tool)
print("그래프 컴파일 완료")

## 5. 그래프 구조 확인 (mermaid)

design doc 3-4절과 비교해서 다른 점이 없는지 눈으로 확인.

In [ ]:
print(graph.get_graph().draw_mermaid())

## 6. 배선 재확인 — API 키 없이 (선택)

01~04번에서 각자 검증했지만, 넷을 실제로 이어붙였을 때도 문제없는지 한 번 더 가짜 객체로 전체를 돌려본다.

In [ ]:
import importlib
import tests.test_graph_wiring as t
importlib.reload(t)

t.run()
print()
t.run_forced_pass_scenario()

## 7. 실제 실행

API 키가 있어야 여기서부터 의미가 있다. `recursion_limit`은 최악의 경우(H가 2번 재시도)를 감안해 넉넉히 잡는다.

In [ ]:
result = graph.invoke({}, config={"recursion_limit": 40})

print("=== validation_result ===")
print(result["validation_result"])
print("\n=== retry_count ===")
print(result["retry_count"])

## 8. 보고서 저장

In [ ]:
from pathlib import Path

output_dir = Path("../output")
output_dir.mkdir(exist_ok=True)
report_path = output_dir / "final_report.md"
report_path.write_text(result["final_report"], encoding="utf-8")

print(f"저장 완료: {report_path}")
print("\n--- 미리보기 (앞 1000자) ---")
print(result["final_report"][:1000])